In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from plasma_core.engine.Dataset import DatasetSpecification, Dataset
from plasma_core.config import DATABASE_CONFIG, META_DATABASE_CONFIG
from plasma_core.engine.Database import PlasmaDatabase
import datetime
import pandas as pd
import numpy as np

In [ ]:
db = PlasmaDatabase(DATABASE_CONFIG, META_DATABASE_CONFIG)


await db.init()

ds_spec = DatasetSpecification(
    parameter_ids = [8],
    date_min = datetime.date(2025,6,1),
    date_max = datetime.date(2025,10,15),
    age_min = 0,
    age_max = 85,
    measure_unit_transform = True,
    math_log = True
)

dataset = Dataset(db, ds_spec)

whole_query, params, columns = dataset.get_SQL_query_params_and_columns(['patient_id'])

whole_query = f'''
    SELECT *
    FROM ({whole_query})
    WHERE lab_id = 1 and parameter_id = 8
'''

# Exclude some parameters except the major data source
raw_query = whole_query + ' AND NOT (lab_id IN (1,10) AND parameter_id=14) AND NOT (lab_id=10 AND parameter_id=49) AND NOT (lab_id=9 AND parameter_id=48)'

first_filter_query = f'''
    SELECT *
    FROM ({raw_query})
    WHERE patient_id NOT IN (
        SELECT DISTINCT patient_id
        FROM (
            SELECT patient_id, 
                toYear(date) AS year,
                count(DISTINCT date) AS count
            FROM ({raw_query})
            GROUP BY patient_id, year
        )
        WHERE count > 2
    )
'''

query = f'''
    WITH percentiles AS (
        SELECT 
            parameter_id,
            quantileExact(0.0001)(value) AS lower_bound,
            quantileExact(0.9999)(value) AS upper_bound
        FROM ({first_filter_query}) AS base
        GROUP BY parameter_id
    )
    SELECT date, value
    FROM ({first_filter_query}) AS base
    JOIN percentiles ON base.parameter_id = percentiles.parameter_id
    WHERE base.value BETWEEN percentiles.lower_bound AND percentiles.upper_bound
'''

In [ ]:
result = await db.execute_q(query, params)
df = pd.DataFrame(result)
df.to_csv('data/crp_real_data.tsv', sep='\t', index=False)

In [ ]:
await db.kill()